In [1]:
ENV["CLIMACOMMS_DEVICE"] = "CUDA"
ENV["CLIMA_NAME_CUDA_KERNELS_FROM_STACK_TRACE"] = "true"

using ClimaCore: Geometry, Operators, MatrixFields
import ClimaCore
import LazyBroadcast: lazy

# Alternatively, we could use Vec₁₂₃, Vec³, etc., if that is more readable.
const C1 = Geometry.Covariant1Vector
const C2 = Geometry.Covariant2Vector
const C12 = Geometry.Covariant12Vector
const C3 = Geometry.Covariant3Vector
const C123 = Geometry.Covariant123Vector
const CT1 = Geometry.Contravariant1Vector
const CT2 = Geometry.Contravariant2Vector
const CT12 = Geometry.Contravariant12Vector
const CT3 = Geometry.Contravariant3Vector
const CT123 = Geometry.Contravariant123Vector
const UVW = Geometry.UVWVector

const divₕ = Operators.Divergence()
const wdivₕ = Operators.WeakDivergence()
const split_divₕ = Operators.SplitDivergence()
const gradₕ = Operators.Gradient()
const wgradₕ = Operators.WeakGradient()
const curlₕ = Operators.Curl()
const wcurlₕ = Operators.WeakCurl()

const ᶜinterp = Operators.InterpolateF2C()
const ᶜdivᵥ = Operators.DivergenceF2C()
const ᶜgradᵥ = Operators.GradientF2C()

# Tracers do not have advective fluxes through the top and bottom cell faces.
const ᶜadvdivᵥ = Operators.DivergenceF2C(
    bottom=Operators.SetValue(CT3(0)),
    top=Operators.SetValue(CT3(0)),
)

# Subsidence has extrapolated tendency at the top, and has no flux at the bottom.
# TODO: This is not accurate and causes some issues at the domain top.
const ᶜsubdivᵥ = Operators.DivergenceF2C(
    bottom=Operators.SetValue(CT3(0)),
    top=Operators.Extrapolate(),
)

# Precipitation has no flux at the top, but it has free outflow at the bottom.
const ᶜprecipdivᵥ = Operators.DivergenceF2C(top=Operators.SetValue(CT3(0)))

const ᶠright_bias = Operators.RightBiasedC2F() # for free outflow in ᶜprecipdivᵥ
const ᶜleft_bias = Operators.LeftBiasedF2C()
const ᶜright_bias = Operators.RightBiasedF2C()

# TODO: Implement proper extrapolation instead of simply reusing the first
# interior value at the surface.
const ᶠinterp = Operators.InterpolateC2F(
    bottom=Operators.Extrapolate(),
    top=Operators.Extrapolate(),
)
const ᶠwinterp = Operators.WeightedInterpolateC2F(
    bottom=Operators.Extrapolate(),
    top=Operators.Extrapolate(),
)

# TODO: Replace these boundary conditions with NaN's, since they are
# meaningless and we only need to specify them in order to be able to
# materialize broadcasts. Any effect these boundary conditions have on the
# boundary values of Y.f.u₃ is overwritten when we call set_velocity_at_surface!.
# Ideally, we would enforce the boundary conditions on Y.f.u₃ by filtering it
# immediately after adding the tendency to it. However, this is not currently
# possible because our implicit solver is unable to handle filtering, which is
# why these boundary conditions are 0's rather than NaN's.
const ᶠgradᵥ = Operators.GradientC2F(
    bottom=Operators.SetGradient(C3(0)),
    top=Operators.SetGradient(C3(0)),
)
const ᶠcurlᵥ = Operators.CurlC2F(
    bottom=Operators.SetCurl(CT12(0, 0)),
    top=Operators.SetCurl(CT12(0, 0)),
)
const upwind_biased_grad = Operators.UpwindBiasedGradient()
const ᶠupwind1 = Operators.UpwindBiasedProductC2F()
const ᶠupwind3 = Operators.Upwind3rdOrderBiasedProductC2F(
    bottom=Operators.ThirdOrderOneSided(),
    top=Operators.ThirdOrderOneSided(),
)
@static if pkgversion(ClimaCore) ≥ v"0.14.22"
    const ᶠlin_vanleer = Operators.LinVanLeerC2F(
        bottom=Operators.FirstOrderOneSided(),
        top=Operators.FirstOrderOneSided(),
        constraint=Operators.MonotoneLocalExtrema(), # (Mono5)
    )
end

const ᶜinterp_matrix = MatrixFields.operator_matrix(ᶜinterp)
const ᶜleft_bias_matrix = MatrixFields.operator_matrix(ᶜleft_bias)
const ᶜright_bias_matrix = MatrixFields.operator_matrix(ᶜright_bias)
const ᶜdivᵥ_matrix = MatrixFields.operator_matrix(ᶜdivᵥ)
const ᶜadvdivᵥ_matrix = MatrixFields.operator_matrix(ᶜadvdivᵥ)
const ᶜprecipdivᵥ_matrix = MatrixFields.operator_matrix(ᶜprecipdivᵥ)
const ᶠright_bias_matrix = MatrixFields.operator_matrix(ᶠright_bias)
const ᶠinterp_matrix = MatrixFields.operator_matrix(ᶠinterp)
const ᶠwinterp_matrix = MatrixFields.operator_matrix(ᶠwinterp)
const ᶠgradᵥ_matrix = MatrixFields.operator_matrix(ᶠgradᵥ)
const ᶠupwind1_matrix = MatrixFields.operator_matrix(ᶠupwind1)
const ᶠupwind3_matrix = MatrixFields.operator_matrix(ᶠupwind3)

# Helper functions to extract components of vectors
u_component(u::Geometry.LocalVector) = u.u
v_component(u::Geometry.LocalVector) = u.v
w_component(u::Geometry.LocalVector) = u.w

include(
    joinpath(pkgdir(ClimaCore), "test", "TestUtilities", "TestUtilities.jl"),
);
import .TestUtilities as TU;
using Test
using ClimaComms
ClimaComms.@import_required_backends
using StaticArrays, IntervalSets
import BenchmarkTools
import StatsBase
import DataStructures
using ClimaCore.Geometry: ⊗
import ClimaCore.DataLayouts

FT = Float32
horizontal_layout_type = ClimaCore.DataLayouts.IJFH
z_elems = 63
helem = 30
Nq = 4
cspace = TU.CenterExtrudedFiniteDifferenceSpace(FT; zelem=z_elems, helem, Nq, horizontal_layout_type)
fspace = ClimaCore.Spaces.FaceExtrudedFiniteDifferenceSpace(cspace)
import ClimaCore: Domains, Meshes, Spaces, Fields, Operators, Topologies
import ClimaCore.Domains: Geometry

using BenchmarkTools
field_vars(::Type{FT}) where {FT} = (;
    x=FT(1),
    uₕ=Geometry.Covariant12Vector(FT(0), FT(0)),
    uₕ2=Geometry.Covariant12Vector(FT(0), FT(0)),
    curluₕ=Geometry.Contravariant12Vector(FT(0), FT(0)),
    w=Geometry.Covariant3Vector(FT(0)),
    contra3=Geometry.Contravariant3Vector(FT(0)),
    y=FT(1),
    D=FT(0),
    U=FT(0),
    ∇x=Geometry.Covariant3Vector(FT(0)),
    ᶠu³=Geometry.Contravariant3Vector(FT(0)),
    ᶠuₕ³=Geometry.Contravariant3Vector(FT(1)),
    ᶠw=Geometry.Covariant3Vector(FT(0)),
)
cfield = fill(field_vars(FT), cspace)
cfield2 = fill(field_vars(FT), cspace)
ffield = fill(field_vars(FT), fspace)
# @. ᶠlin_vanleer(ffield.ᶠu³, cfield.x, 0.1f0)
ᶜJ = Fields.local_geometry_field(axes(cfield)).J
ᶠJ = Fields.local_geometry_field(axes(ffield)).J

┌ Warning: CUDA runtime library `libcublasLt.so.13` was loaded from a system path, `/usr/local/cuda/lib64/libcublasLt.so.13`.
│ This may cause errors.
│ 
│ If you're running under a profiler, this situation is expected. Otherwise,
│ ensure that your library path environment variable (e.g., `PATH` on Windows
│ or `LD_LIBRARY_PATH` on Linux) does not include CUDA library paths.
│ 
│ In any other case, please file an issue.
└ @ CUDA ~/.julia/packages/CUDA/TPbi4/src/initialization.jl:218
┌ Warning: CUDA runtime library `libnvJitLink.so.13` was loaded from a system path, `/usr/local/cuda/lib64/libnvJitLink.so.13`.
│ This may cause errors.
│ 
│ If you're running under a profiler, this situation is expected. Otherwise,
│ ensure that your library path environment variable (e.g., `PATH` on Windows
│ or `LD_LIBRARY_PATH` on Linux) does not include CUDA library paths.
│ 
│ In any other case, please file an issue.
└ @ CUDA ~/.julia/packages/CUDA/TPbi4/src/initialization.jl:218
┌ Warning: CUDA runt

Float32-valued Field:
  Float32[0.132472, 0.132472, 0.132472, 0.132472, 0.132472, 0.132472, 0.132472, 0.132472, 0.132472, 0.132472  …  0.132472, 0.132471, 0.132472, 0.132471, 0.132472, 0.132471, 0.132472, 0.132471, 0.132472, 0.132472]

In [2]:
bi_fs = (fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), fspace))

bi_cs = (fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace),
       fill(ClimaCore.MatrixFields.BidiagonalMatrixRow(0.0f0, 0.0f0), cspace))

di_fs = (fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), fspace))

di_cs = (fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace))

di_cs2 = (fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace))

di_cs3 = (fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace),
       fill(ClimaCore.MatrixFields.DiagonalMatrixRow(0.0f0), cspace))

tri = fill(ClimaCore.MatrixFields.TridiagonalMatrixRow(0.0f0, 0.0f0, 0.0f0), fspace)
tri2 = fill(ClimaCore.MatrixFields.TridiagonalMatrixRow(0.0f0, 0.0f0, 0.0f0), cspace)

ClimaCore.MatrixFields.TridiagonalMatrixRow{Float32}-valued Field whose first column corresponds to the Square matrix
 0.0  0.0   ⋅    ⋅    ⋅    ⋅    ⋅    ⋅   …   ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
 0.0  0.0  0.0   ⋅    ⋅    ⋅    ⋅    ⋅       ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅   0.0  0.0  0.0   ⋅    ⋅    ⋅    ⋅       ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅   0.0  0.0  0.0   ⋅    ⋅    ⋅       ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅   0.0  0.0  0.0   ⋅    ⋅       ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅   0.0  0.0  0.0   ⋅   …   ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅   0.0  0.0  0.0      ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅    ⋅   0.0  0.0      ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅   0.0      ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅       ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅   …   ⋅    ⋅    ⋅    ⋅    ⋅    ⋅    ⋅ 
  ⋅    ⋅    ⋅    ⋅    ⋅ 

In [3]:
res = CUDA.@profile trace=true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2]) + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4]) + (bi_fs[5] * di_cs[5] * bi_cs[5]) + (bi_fs[6] * di_cs[6] * bi_cs[6]) + (bi_fs[7] * di_cs[7] * bi_cs[7]) + (bi_fs[8] * di_cs[8] * bi_cs[8]) + (bi_fs[9] * di_cs[9] * bi_cs[9])) - tri
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2]) + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4]) + (bi_fs[5] * di_cs[5] * bi_cs[5]) + (bi_fs[6] * di_cs[6] * bi_cs[6]) + (bi_fs[7] * di_cs[7] * bi_cs[7]) + (bi_fs[8] * di_cs[8] * bi_cs[8])) - tri
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2]) + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4]) + (bi_fs[5] * di_cs[5] * bi_cs[5]) + (bi_fs[6] * di_cs[6] * bi_cs[6]) + (bi_fs[7] * di_cs[7] * bi_cs[7])) - tri
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2]) + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4]) + (bi_fs[5] * di_cs[5] * bi_cs[5]) + (bi_fs[6] * di_cs[6] * bi_cs[6])) - tri
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2]) + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4]) + (bi_fs[5] * di_cs[5] * bi_cs[5])) - tri
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2]) + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4])) - tri
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2]) + (bi_fs[3] * di_cs[3] * bi_cs[3])) - tri
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])) - tri
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1])) - tri
end

res

Profiler ran for 136.25 s, capturing 1082 events.

Host-side activity: calling CUDA APIs took 12.69 ms (0.01% of the trace)
┌──────┬───────────┬───────────┬─────────────────────┬───────────────────────────┐
│   ID │     Start │      Time │ Name                │                   Details │
├──────┼───────────┼───────────┼─────────────────────┼───────────────────────────┤
│    6 │ 957.57 ms │ 540.02 µs │ cuMemAlloc          │ 63.281 MiB, device memory │
│  151 │   29.67 s │   29.8 µs │ cuCtxSynchronize    │                         - │
│  155 │   29.67 s │ 172.62 µs │ cuModuleLoadDataEx  │                         - │
│  160 │   29.67 s │ 201.94 µs │ cuModuleGetFunction │                         - │
│  164 │   29.67 s │  52.69 µs │ cuLaunchKernel      │                         - │
│  169 │   30.34 s │ 159.26 µs │ cuMemGetInfo        │                         - │
│  174 │   30.34 s │ 605.11 µs │ cuMemAlloc          │ 63.281 MiB, device memory │
│  294 │   55.41 s │   1.19 µs │ cuCtxSetCurre

In [4]:
res.device

Row,id,start,stop,name,device,context,stream,grid,block,registers,shared_mem,local_mem,size,details
,Int64,Float64,Float64,Abstract…,Int64,Int64,Int64,CuDim3?,CuDim3?,Int64?,NamedTup…?,NamedTup…?,UInt64?,String?
1,34,1.77369e9,1.77369e9,[copy pageable to device memory],0,1,13,missing,missing,missing,missing,missing,8,missing
2,201,1.77369e9,1.77369e9,_FILE__julia_packages_CUDA_TPbi4_src_profile_jl_L66,0,1,13,"CuDim3(0x00000004, 0x00000004, 0x00001518)","CuDim3(0x00000040, 0x00000001, 0x00000001)",58,"(static = 0, dynamic = 0)","(thread = 0, total = 399900672)",missing,missing
3,360,1.77369e9,1.77369e9,_FILE__julia_packages_CUDA_TPbi4_src_profile_jl_L66,0,1,13,"CuDim3(0x00000004, 0x00000004, 0x00001518)","CuDim3(0x00000040, 0x00000001, 0x00000001)",54,"(static = 0, dynamic = 0)","(thread = 0, total = 375128064)",missing,missing
4,504,1.77369e9,1.77369e9,_FILE__julia_packages_CUDA_TPbi4_src_profile_jl_L66,0,1,13,"CuDim3(0x00000004, 0x00000004, 0x00001518)","CuDim3(0x00000040, 0x00000001, 0x00000001)",52,"(static = 0, dynamic = 0)","(thread = 0, total = 350355456)",missing,missing
5,636,1.77369e9,1.77369e9,_FILE__julia_packages_CUDA_TPbi4_src_profile_jl_L66,0,1,13,"CuDim3(0x00000004, 0x00000004, 0x00001518)","CuDim3(0x00000040, 0x00000001, 0x00000001)",48,"(static = 0, dynamic = 0)","(thread = 0, total = 325582848)",missing,missing
6,756,1.77369e9,1.77369e9,_FILE__julia_packages_CUDA_TPbi4_src_profile_jl_L66,0,1,13,"CuDim3(0x00000004, 0x00000004, 0x00001518)","CuDim3(0x00000040, 0x00000001, 0x00000001)",48,"(static = 0, dynamic = 0)","(thread = 0, total = 300810240)",missing,missing
7,864,1.77369e9,1.77369e9,_FILE__julia_packages_CUDA_TPbi4_src_profile_jl_L66,0,1,13,"CuDim3(0x00000004, 0x00000004, 0x00001518)","CuDim3(0x00000040, 0x00000001, 0x00000001)",46,"(static = 0, dynamic = 0)","(thread = 0, total = 276037632)",missing,missing
8,956,1.77369e9,1.77369e9,_FILE__julia_packages_CUDA_TPbi4_src_profile_jl_L66,0,1,13,"CuDim3(0x00000004, 0x00000004, 0x00001518)","CuDim3(0x00000040, 0x00000001, 0x00000001)",40,"(static = 0, dynamic = 0)","(thread = 0, total = 251265024)",missing,missing
9,1040,1.77369e9,1.77369e9,_FILE__julia_packages_CUDA_TPbi4_src_profile_jl_L66,0,1,13,"CuDim3(0x00000004, 0x00000004, 0x00001518)","CuDim3(0x00000040, 0x00000001, 0x00000001)",37,"(static = 0, dynamic = 0)","(thread = 0, total = 226492416)",missing,missing


In [5]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1])) - tri
end

Profiler ran for 792.26 µs, capturing 47 events.

Host-side activity: calling CUDA APIs took 398.4 µs (50.29% of the trace)
┌────┬───────────┬───────────┬────────────────┬───────────────────────────┐
│ ID │     Start │      Time │ Name           │                   Details │
├────┼───────────┼───────────┼────────────────┼───────────────────────────┤
│  6 │  21.46 µs │ 369.07 µs │ cuMemAlloc     │ 63.281 MiB, device memory │
│ 44 │ 445.13 µs │   24.8 µs │ cuLaunchKernel │                         - │
└────┴───────────┴───────────┴────────────────┴───────────────────────────┘

Device-side activity: GPU was busy for 317.34 µs (40.05% of the trace)
┌────┬───────────┬───────────┬─────────┬──────────┬──────┬─────────────────────────────────────────────────────┐
│ ID │     Start │      Time │ Threads │   Blocks │ Regs │ Name                                                │
├────┼───────────┼───────────┼─────────┼──────────┼──────┼─────────────────────────────────────────────────────┤
│ 44 │ 47

In [6]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])) - tri
end

Profiler ran for 1.15 ms, capturing 59 events.

Host-side activity: calling CUDA APIs took 390.05 µs (33.96% of the trace)
┌────┬───────────┬───────────┬────────────────┬───────────────────────────┐
│ ID │     Start │      Time │ Name           │                   Details │
├────┼───────────┼───────────┼────────────────┼───────────────────────────┤
│  6 │   21.7 µs │ 358.58 µs │ cuMemAlloc     │ 63.281 MiB, device memory │
│ 56 │ 432.97 µs │  27.42 µs │ cuLaunchKernel │                         - │
└────┴───────────┴───────────┴────────────────┴───────────────────────────┘

Device-side activity: GPU was busy for 684.02 µs (59.55% of the trace)
┌────┬───────────┬───────────┬─────────┬──────────┬──────┬─────────────────────────────────────────────────────┐
│ ID │     Start │      Time │ Threads │   Blocks │ Regs │ Name                                                │
├────┼───────────┼───────────┼─────────┼──────────┼──────┼─────────────────────────────────────────────────────┤
│ 56 │ 460

In [7]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3])) - tri
end

Profiler ran for 1.55 ms, capturing 71 events.

Host-side activity: calling CUDA APIs took 450.37 µs (29.00% of the trace)
┌────┬───────────┬───────────┬────────────────┬───────────────────────────┐
│ ID │     Start │      Time │ Name           │                   Details │
├────┼───────────┼───────────┼────────────────┼───────────────────────────┤
│  6 │  37.43 µs │ 417.71 µs │ cuMemAlloc     │ 63.281 MiB, device memory │
│ 68 │ 517.13 µs │  25.99 µs │ cuLaunchKernel │                         - │
└────┴───────────┴───────────┴────────────────┴───────────────────────────┘

Device-side activity: GPU was busy for 1.01 ms (64.75% of the trace)
┌────┬───────────┬─────────┬─────────┬──────────┬──────┬─────────────────────────────────────────────────────┐
│ ID │     Start │    Time │ Threads │   Blocks │ Regs │ Name                                                │
├────┼───────────┼─────────┼─────────┼──────────┼──────┼─────────────────────────────────────────────────────┤
│ 68 │ 543.59 µs │

In [8]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4])) - tri
end

Profiler ran for 2.47 ms, capturing 83 events.

Host-side activity: calling CUDA APIs took 1.07 ms (43.29% of the trace)
┌────┬──────────┬──────────┬────────────────┬───────────────────────────┐
│ ID │    Start │     Time │ Name           │                   Details │
├────┼──────────┼──────────┼────────────────┼───────────────────────────┤
│  6 │ 27.42 µs │  1.03 ms │ cuMemAlloc     │ 63.281 MiB, device memory │
│ 80 │  1.12 ms │ 34.09 µs │ cuLaunchKernel │                         - │
└────┴──────────┴──────────┴────────────────┴───────────────────────────┘

Device-side activity: GPU was busy for 1.33 ms (53.64% of the trace)
┌────┬─────────┬─────────┬─────────┬──────────┬──────┬─────────────────────────────────────────────────────┐
│ ID │   Start │    Time │ Threads │   Blocks │ Regs │ Name                                                │
├────┼─────────┼─────────┼─────────┼──────────┼──────┼─────────────────────────────────────────────────────┤
│ 80 │ 1.14 ms │ 1.33 ms │      64 │ 4

In [9]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4])
        + (bi_fs[5] * di_cs[5] * bi_cs[5])) - tri
end

Profiler ran for 2.26 ms, capturing 95 events.

Host-side activity: calling CUDA APIs took 575.54 µs (25.45% of the trace)
┌────┬───────────┬──────────┬────────────────┬───────────────────────────┐
│ ID │     Start │     Time │ Name           │                   Details │
├────┼───────────┼──────────┼────────────────┼───────────────────────────┤
│  6 │   45.3 µs │ 537.4 µs │ cuMemAlloc     │ 63.281 MiB, device memory │
│ 92 │ 658.51 µs │ 29.33 µs │ cuLaunchKernel │                         - │
└────┴───────────┴──────────┴────────────────┴───────────────────────────┘

Device-side activity: GPU was busy for 1.57 ms (69.37% of the trace)
┌────┬───────────┬─────────┬─────────┬──────────┬──────┬─────────────────────────────────────────────────────┐
│ ID │     Start │    Time │ Threads │   Blocks │ Regs │ Name                                                │
├────┼───────────┼─────────┼─────────┼──────────┼──────┼─────────────────────────────────────────────────────┤
│ 92 │ 688.31 µs │ 1.57 

In [10]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4])
        + (bi_fs[5] * di_cs[5] * bi_cs[5]) + (bi_fs[6] * di_cs[6] * bi_cs[6])) - tri
end

Profiler ran for 3.03 ms, capturing 107 events.

Host-side activity: calling CUDA APIs took 1.03 ms (33.84% of the trace)
┌─────┬──────────┬──────────┬────────────────┬───────────────────────────┐
│  ID │    Start │     Time │ Name           │                   Details │
├─────┼──────────┼──────────┼────────────────┼───────────────────────────┤
│   6 │ 61.99 µs │ 992.3 µs │ cuMemAlloc     │ 63.281 MiB, device memory │
│ 104 │  1.13 ms │ 26.23 µs │ cuLaunchKernel │                         - │
└─────┴──────────┴──────────┴────────────────┴───────────────────────────┘

Device-side activity: GPU was busy for 1.87 ms (61.77% of the trace)
┌─────┬─────────┬─────────┬─────────┬──────────┬──────┬─────────────────────────────────────────────────────┐
│  ID │   Start │    Time │ Threads │   Blocks │ Regs │ Name                                                │
├─────┼─────────┼─────────┼─────────┼──────────┼──────┼─────────────────────────────────────────────────────┤
│ 104 │ 1.16 ms │ 1.87 ms │ 

In [11]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4])
        + (bi_fs[5] * di_cs[5] * bi_cs[5]) + (bi_fs[6] * di_cs[6] * bi_cs[6])
        + (bi_fs[7] * di_cs[7] * bi_cs[7])) - tri
end

Profiler ran for 2.84 ms, capturing 119 events.

Host-side activity: calling CUDA APIs took 467.78 µs (16.47% of the trace)
┌─────┬───────────┬───────────┬────────────────┬───────────────────────────┐
│  ID │     Start │      Time │ Name           │                   Details │
├─────┼───────────┼───────────┼────────────────┼───────────────────────────┤
│   6 │  87.02 µs │ 431.54 µs │ cuMemAlloc     │ 63.281 MiB, device memory │
│ 116 │ 602.01 µs │   26.7 µs │ cuLaunchKernel │                         - │
└─────┴───────────┴───────────┴────────────────┴───────────────────────────┘

Device-side activity: GPU was busy for 2.21 ms (77.67% of the trace)
┌─────┬───────────┬─────────┬─────────┬──────────┬──────┬─────────────────────────────────────────────────────┐
│  ID │     Start │    Time │ Threads │   Blocks │ Regs │ Name                                                │
├─────┼───────────┼─────────┼─────────┼──────────┼──────┼─────────────────────────────────────────────────────┤
│ 116 │ 

In [12]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4])
        + (bi_fs[5] * di_cs[5] * bi_cs[5]) + (bi_fs[6] * di_cs[6] * bi_cs[6])
        + (bi_fs[7] * di_cs[7] * bi_cs[7]) + (bi_fs[8] * di_cs[8] * bi_cs[8])) - tri
end

Profiler ran for 3.7 ms, capturing 131 events.

Host-side activity: calling CUDA APIs took 1.02 ms (27.56% of the trace)
┌─────┬──────────┬───────────┬────────────────┬───────────────────────────┐
│  ID │    Start │      Time │ Name           │                   Details │
├─────┼──────────┼───────────┼────────────────┼───────────────────────────┤
│   6 │ 71.76 µs │ 982.52 µs │ cuMemAlloc     │ 63.281 MiB, device memory │
│ 128 │  1.14 ms │  27.18 µs │ cuLaunchKernel │                         - │
└─────┴──────────┴───────────┴────────────────┴───────────────────────────┘

Device-side activity: GPU was busy for 2.53 ms (68.36% of the trace)
┌─────┬─────────┬─────────┬─────────┬──────────┬──────┬─────────────────────────────────────────────────────┐
│  ID │   Start │    Time │ Threads │   Blocks │ Regs │ Name                                                │
├─────┼─────────┼─────────┼─────────┼──────────┼──────┼─────────────────────────────────────────────────────┤
│ 128 │ 1.17 ms │ 2.53 

In [13]:
CUDA.@profile trace = true begin
    @. ((bi_fs[1] * di_cs[1] * bi_cs[1]) + (bi_fs[2] * di_cs[2] * bi_cs[2])
        + (bi_fs[3] * di_cs[3] * bi_cs[3]) + (bi_fs[4] * di_cs[4] * bi_cs[4])
        + (bi_fs[5] * di_cs[5] * bi_cs[5]) + (bi_fs[6] * di_cs[6] * bi_cs[6])
        + (bi_fs[7] * di_cs[7] * bi_cs[7]) + (bi_fs[8] * di_cs[8] * bi_cs[8])
        + (bi_fs[9] * di_cs[9] * bi_cs[9])) - tri
end

Profiler ran for 3.67 ms, capturing 143 events.

Host-side activity: calling CUDA APIs took 560.52 µs (15.28% of the trace)
┌─────┬───────────┬───────────┬────────────────┬───────────────────────────┐
│  ID │     Start │      Time │ Name           │                   Details │
├─────┼───────────┼───────────┼────────────────┼───────────────────────────┤
│   6 │  97.27 µs │ 520.23 µs │ cuMemAlloc     │ 63.281 MiB, device memory │
│ 140 │ 710.25 µs │  29.56 µs │ cuLaunchKernel │                         - │
└─────┴───────────┴───────────┴────────────────┴───────────────────────────┘

Device-side activity: GPU was busy for 2.92 ms (79.67% of the trace)
┌─────┬───────────┬─────────┬─────────┬──────────┬──────┬─────────────────────────────────────────────────────┐
│  ID │     Start │    Time │ Threads │   Blocks │ Regs │ Name                                                │
├─────┼───────────┼─────────┼─────────┼──────────┼──────┼─────────────────────────────────────────────────────┤
│ 140 │ 